In [1]:
import asreview
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import synergy_dataset as sd
from asreview.models.balancers import Balanced
from asreview.models.classifiers import NaiveBayes
from asreview.models.feature_extractors import Tfidf
from asreview.models.queriers import Max
from matplotlib.backends.backend_pdf import PdfPages
from scipy.interpolate import interp1d

# Path to your SQLite3 database
db_path = ""
data_path = "./data/"

In [3]:
def pad_labels(labels, num_priors, num_records):
    return pd.Series(
        labels.tolist() + np.zeros(num_records - len(labels) - num_priors).tolist()
    )


def n_query_extreme(results, n_records):
    if n_records >= 10000:
        if len(results) >= 10000:
            return 10**5  # finish the run
        if len(results) >= 1000:
            return 1000
        elif len(results) >= 100:
            return 25
        else:
            return 1
    else:
        if len(results) >= 1000:
            return 100
        elif len(results) >= 100:
            return 5
        else:
            return 1

# One Study, One Dataset


In [ ]:
study = {
    "dataset_id": "Donners_2021",
    "prior_inclusions": [81],
    "prior_exclusions": [218, 143, 223, 181, 86, 249, 105],
}
X = sd.Dataset(study["dataset_id"]).to_frame().reset_index()
priors = study["prior_inclusions"] + study["prior_exclusions"]
labels = X["label_included"]
study_1_name = "ASReview2-full-tfidf-nb-3"
study_2_name = "ASReview2-full-nb-1"

In [ ]:
study = optuna.load_study(study_name=study_1_name, storage=db_path)
params = study.best_trial.params

alc = asreview.ActiveLearningCycle(
    querier=Max(),
    classifier=NaiveBayes(alpha=params["nb__alpha"]),
    balancer=Balanced(ratio=params["ratio"]),
    feature_extractor=Tfidf(
        stop_words=None,
        ngram_range=(1, 2),
        sublinear_tf=True,
        max_df=params["tfidf__max_df"],
        min_df=params["tfidf__min_df"],
    ),
)
simulate_1 = asreview.Simulate(
    X=X,
    labels=labels,
    cycles=[alc],
)
# Set priors
simulate_1.label(priors)
# Start simulation
simulate_1.review()


df_1 = simulate_1._results.dropna(axis=0, subset="training_set")
labels_1 = pad_labels(df_1["label"].reset_index(drop=True), len(priors), len(X))
recall_1 = labels_1.cumsum()

In [ ]:
study = optuna.load_study(study_name=study_2_name, storage=db_path)
params = study.best_trial.params

alc = asreview.ActiveLearningCycle(
    querier=Max(),
    classifier=NaiveBayes(alpha=params["alpha"]),
    balancer=Balanced(ratio=params["ratio"]),
    feature_extractor=Tfidf(
        stop_words=None,
        ngram_range=(1, 2),
        sublinear_tf=True,
        max_df=params["tfidf__max_df"],
        min_df=params["tfidf__min_df"],
    ),
)
simulate_2 = asreview.Simulate(
    X=X,
    labels=labels,
    cycles=[alc],
)
# Set priors
simulate_2.label(priors)
# Start simulation
simulate_2.review()

df_2 = simulate_2._results.dropna(axis=0, subset="training_set")
labels_2 = pad_labels(df_2["label"].reset_index(drop=True), len(priors), len(X))
recall_2 = labels_2.cumsum()

In [ ]:
alc = asreview.ActiveLearningCycle(
    querier=Max(),
    classifier=NaiveBayes(alpha=3.822),
    balancer=Balanced(ratio=1.2),
    feature_extractor=Tfidf(stop_words="english", ngram_range=(1, 1)),
)
simulate_old = asreview.Simulate(
    X=X,
    labels=labels,
    cycles=[alc],
)
# Set priors
simulate_old.label(priors)
# Start simulation
simulate_old.review()

df_old = simulate_old._results.dropna(axis=0, subset="training_set")
labels_old = pad_labels(df_old["label"].reset_index(drop=True), len(priors), len(X))
recall_old = labels_old.cumsum()

In [ ]:
combined = pd.DataFrame(
    {study_1_name: recall_1, study_2_name: recall_1, "old": recall_old}
)

combined.plot()
plt.savefig("recall_comp.pdf")

# Mean of Multiple Studies, One Dataset


In [ ]:
dataset_name = "Appenzeller-Herzog_2019"
studies = pd.read_json("synergy_studies_validation.jsonl", lines=True)
studies = studies[studies["dataset_id"] == dataset_name]
recalls_ndcg = []
recalls_loss = []
recalls_old = []

for _, study in studies.iterrows():
    X = sd.Dataset(study["dataset_id"]).to_frame().reset_index()
    priors = study["prior_inclusions"] + study["prior_exclusions"]
    labels = X["label_included"]
    study_name = "ASReview2-full-tfidf-nb-3"
    study = optuna.load_study(study_name=study_name, storage=db_path)
    params = study.best_trial.params

    alc = asreview.ActiveLearningCycle(
        querier=Max(),
        classifier=NaiveBayes(alpha=params["nb__alpha"]),
        balancer=Balanced(ratio=params["ratio"]),
        feature_extractor=Tfidf(
            stop_words=None,
            ngram_range=(1, 2),
            sublinear_tf=True,
            max_df=params["tfidf__max_df"],
            min_df=params["tfidf__min_df"],
        ),
    )
    simulate_ndcg = asreview.Simulate(
        X=X,
        labels=labels,
        cycles=[alc],
    )
    # Set priors
    simulate_ndcg.label(priors)
    # Start simulation
    simulate_ndcg.review()

    df_ndcg = simulate_ndcg._results.dropna(axis=0, subset="training_set")
    labels_ndcg = pad_labels(
        df_ndcg["label"].reset_index(drop=True), len(priors), len(X)
    )
    recalls_ndcg.append(labels_ndcg.cumsum())

for _, study in studies.iterrows():
    X = sd.Dataset(study["dataset_id"]).to_frame().reset_index()
    priors = study["prior_inclusions"] + study["prior_exclusions"]
    labels = X["label_included"]
    study_name = "ASReview2-full-nb-1"
    study = optuna.load_study(study_name=study_name, storage=db_path)
    params = study.best_trial.params

    alc = asreview.ActiveLearningCycle(
        querier=Max(),
        classifier=NaiveBayes(alpha=params["alpha"]),
        balancer=Balanced(ratio=params["ratio"]),
        feature_extractor=Tfidf(
            stop_words=None,
            ngram_range=(1, 2),
            sublinear_tf=True,
            max_df=params["tfidf__max_df"],
            min_df=params["tfidf__min_df"],
        ),
    )
    simulate_loss = asreview.Simulate(
        X=X,
        labels=labels,
        cycles=[alc],
    )
    # Set priors
    simulate_loss.label(priors)
    # Start simulation
    simulate_loss.review()

    df_loss = simulate_loss._results.dropna(axis=0, subset="training_set")
    labels_loss = pad_labels(
        df_loss["label"].reset_index(drop=True), len(priors), len(X)
    )
    recalls_loss.append(labels_loss.cumsum())

for _, study in studies.iterrows():
    X = sd.Dataset(study["dataset_id"]).to_frame().reset_index()
    priors = study["prior_inclusions"] + study["prior_exclusions"]
    labels = X["label_included"]
    alc = asreview.ActiveLearningCycle(
        querier=Max(),
        classifier=NaiveBayes(alpha=3.822),
        balancer=Balanced(ratio=3),
        feature_extractor=Tfidf(stop_words="english", ngram_range=(1, 1)),
    )
    simulate_old = asreview.Simulate(
        X=X,
        labels=labels,
        cycles=[alc],
    )
    # Set priors
    simulate_old.label(priors)
    # Start simulation
    simulate_old.review()

    df_old = simulate_old._results.dropna(axis=0, subset="training_set")
    labels_old = pad_labels(df_old["label"].reset_index(drop=True), len(priors), len(X))
    recalls_old.append(labels_old.cumsum())

In [ ]:
arrays = [np.array(x) for x in recalls_loss]
new_recalls_loss = [np.mean(k) for k in zip(*arrays)]

arrays = [np.array(x) for x in recalls_ndcg]
new_recalls_ndcg = [np.mean(k) for k in zip(*arrays)]

arrays = [np.array(x) for x in recalls_old]
new_recalls_old = [np.mean(k) for k in zip(*arrays)]

In [ ]:
combined = pd.DataFrame(
    {"loss": new_recalls_loss, "gain": new_recalls_ndcg, "old": new_recalls_old}
)

combined.plot()
plt.savefig(f"recall_comp_{dataset_name}.pdf")

# Mean of Multiple Studies, All Datasets


In [30]:
studies = pd.read_json("synergy_studies_validation.jsonl", lines=True)
studies_filtered = studies.sort_values("dataset_id").reset_index(drop=True)

recall_files = [
    "recalls_new2_nb.csv",
    "recalls_new2_svm.csv",
    "recalls_old1_nb.csv",
    "recalls_old1_svm.csv",
    "recalls_new2_mxbai_svm.csv",
    "recalls_new2_e5_svm.csv", 
]
recall_types = ["ASR2 Naive Bayes", "ASR2 SVM", "ASR1.6 Naive Bayes", "ASR1.6 SVM", "ASR2 SVM_mxbai", "ASR2 SVM_e5"]

In [31]:
def get_total_relevant(dataset_id):
    if dataset_id in {"Moran_2021_corrected", "Muthu_2021_corrected"}:
        return pd.read_csv(f"../src/datasets/{dataset_id}_shuffled_raw.csv")[
            "label_included"
        ].sum()
    else:
        return sd.Dataset(dataset_id).to_frame()["label_included"].sum()


# Build the dictionary
total_relevant_dict = {
    dataset_id: get_total_relevant(dataset_id)
    for dataset_id in studies_filtered["dataset_id"].unique()
}

In [32]:
recall_dfs = [pd.read_csv(data_path + f) for f in recall_files]

# Add metadata to each DataFrame
for i, df in enumerate(recall_dfs):
    df["dataset_name"] = studies_filtered["dataset_id"].values
    df["Optimization"] = recall_types[i]
    df["prior_inclusions"] = studies_filtered["prior_inclusions"].apply(len)
    df["prior_exclusions"] = studies_filtered["prior_exclusions"].apply(len)
    df["simulation_id"] = df.groupby("dataset_name").cumcount() + 1

# Combine recall data
df_all = pd.concat(recall_dfs, ignore_index=True)

# Melt dataframe
df_all_melted = df_all.melt(
    id_vars=[
        "dataset_name",
        "Optimization",
        "prior_inclusions",
        "prior_exclusions",
        "simulation_id",
    ],
    var_name="step",
    value_name="recall",
).dropna()

# Convert step to numeric
df_all_melted["step"] = df_all_melted["step"].astype(int)

# Calculate total relevant items
df_all_melted["total_relevant"] = df_all_melted["dataset_name"].map(total_relevant_dict)

# Recall normalization
df_all_melted["relative_recall"] = df_all_melted["recall"] / (
    df_all_melted["total_relevant"] - df_all_melted["prior_inclusions"]
)

# Normalize step values to a common scale [0,1]
df_all_melted["relative_step"] = df_all_melted.groupby(
    ["dataset_name", "Optimization"]
)["step"].transform(lambda x: x / x.max())

In [33]:
df_all_melted_srt = df_all_melted.sort_values(
    ["dataset_name", "Optimization", "simulation_id", "step"]
)

# Group and compute diff to get per-step labels
df_all_melted_srt["label"] = (
    df_all_melted_srt.groupby(["dataset_name", "Optimization", "simulation_id"])[
        "recall"
    ]
    .diff()
    .fillna(df_all_melted["recall"])  # First value is just recall itself
    .astype(int)
)

In [38]:
def rank_loss(labels):
    """
    Custom loss that rewards 1s appearing earlier.
    labels: binary list or array, where higher value = better
    """
    labels = np.asarray(labels)
    Nx = len(labels)
    Ny = labels.sum()

    if Ny == 0 or Ny == Nx:
        return 0.0  # edge case: no 1s or all 1s

    cumsum_sum = np.cumsum(labels).sum()
    loss = (Ny * (Nx - (Ny - 1) / 2) - cumsum_sum) / (Ny * (Nx - Ny))
    return loss


# Make sure data is sorted
df_all_melted_srt = df_all_melted.sort_values(
    ["dataset_name", "Optimization", "simulation_id", "step"]
)

# Group and compute diff to get per-step labels
df_all_melted_srt["label"] = (
    df_all_melted_srt.groupby(["dataset_name", "Optimization", "simulation_id"])[
        "recall"
    ]
    .diff()
    .fillna(df_all_melted["recall"])  # First value is just recall itself
    .astype(int)
)

grouped_loss = (
    df_all_melted_srt.groupby(["dataset_name", "Optimization", "simulation_id"])[
        "label"
    ]
    .apply(rank_loss)
    .reset_index(name="loss")
)

# Compute standard deviation of loss per dataset and optimization
std_loss_per_dataset_per_optimization = (
    grouped_loss.groupby(["dataset_name", "Optimization"])["loss"]
    .std()
    .reset_index()
)

# Compute mean std per optimization (averaging across datasets)
std_loss_per_optimization = (
    std_loss_per_dataset_per_optimization.groupby(["Optimization"])["loss"]
    .mean()
    .reset_index()
)
std_loss_per_optimization[['Version', 'Classifier']] = std_loss_per_optimization['Optimization'].str.extract(r'(ASR\d+\.*\d*)\s(.*)')

# Create a pivot table for standard deviation
pivot_std_df = std_loss_per_optimization.pivot(index='Classifier', columns='Version', values='loss')

mean_loss_per_dataset_per_optimization = (
    grouped_loss.groupby(["dataset_name", "Optimization"])["loss"].mean().reset_index()
)

mean_loss_per_optimization = (
    mean_loss_per_dataset_per_optimization.groupby(["Optimization"])["loss"]
    .mean()
    .reset_index()
)
mean_loss_per_optimization[['Version', 'Classifier']] = mean_loss_per_optimization['Optimization'].str.extract(r'(ASR\d+\.*\d*)\s(.*)')

# Create a pivot table
pivot_df = mean_loss_per_optimization.pivot(index='Classifier', columns='Version', values='loss')
print("---MEAN---")
print(pivot_df)
print("\n----SD----")
print(pivot_std_df)

---MEAN---
Version        ASR1.6      ASR2
Classifier                     
Naive Bayes  0.080632  0.064839
SVM          0.081293  0.062001
SVM_e5            NaN  0.064023
SVM_mxbai         NaN  0.061021

----SD----
Version        ASR1.6      ASR2
Classifier                     
Naive Bayes  0.007264  0.004964
SVM          0.007194  0.004007
SVM_e5            NaN  0.004240
SVM_mxbai         NaN  0.004648


In [35]:
# Interpolation of normalized recall to a common x-axis
x_new = np.linspace(0, 1, 1000)  # 1000 evenly spaced points for smooth curves

interpolated_data = []
for (dataset, recall_type), group in df_all_melted.groupby(
    ["dataset_name", "Optimization"]
):
    group = (
        group.sort_values("relative_step")
        .groupby("relative_step", as_index=False)["relative_recall"]
        .mean()
    )

    x, y = group["relative_step"].values, group["relative_recall"].values
    f = interp1d(x, y, kind="linear", fill_value="extrapolate")
    y_new = f(x_new)

    interpolated_data.extend(
        {
            "dataset_name": dataset,
            "Optimization": recall_type,
            "relative_step": step,
            "relative_recall": recall,
        }
        for step, recall in zip(x_new, y_new)
    )

# Create DataFrame from interpolated results
df_interpolated = pd.DataFrame(interpolated_data)

# Compute the mean recall per dataset and recall type
df_grouped = (
    df_all_melted.groupby(["dataset_name", "Optimization", "relative_step"])[
        "relative_recall"
    ]
    .mean()
    .reset_index()
)

In [36]:
# Define grid size
rows, cols = 5, 5
datasets = df_grouped["dataset_name"].unique()

# Create PDF to store all plots
with PdfPages("recall_comparison_old_new.pdf") as pdf:
    fig, axes = plt.subplots(rows, cols, figsize=(20, 20))  # Adjust figure size
    axes = axes.flatten()  # Flatten for easy iteration

    for i, dataset in enumerate(datasets):
        ax = axes[i]  # Get subplot axis
        df_subset = df_grouped[df_grouped["dataset_name"] == dataset]

        # Plot mean recall with standard deviation as shaded area
        sns.lineplot(
            data=df_subset,
            x="relative_step",
            y="relative_recall",
            hue="Optimization",
            ax=ax,
            legend=True,
        )

        ax.set_xlabel("Proportion of Documents")
        ax.set_ylabel("Mean Recall")
        ax.set_title(f"{dataset}")

    # Adjust layout and save
    plt.suptitle("Mean Relative Recalls of Naive Bayes per SYNERGY Dataset")
    plt.tight_layout(rect=[0, 0.03, 1, 0.99])
    pdf.savefig(fig)  # Save the entire grid as one page in the PDF
    plt.close(fig)  # Close figure to free memory
